# INF201 – Deliverable 3: A mini-RPG battle simulator

**Authors:** Mathias Gundersen & Jakub Mermer
**GitHub repository:** https://github.com/saltonese/Del_3-G6_5.git

In this notebook we present the classes that make up our battle simulator, and use them to simulate battles between different types of characters, with and without weapons. The reusable code lives in separate modules; the notebook is only used for presentation and experiments.

| Module | Classes | Purpose |
|---|---|---|
| `dice.py` | `Dice` | Randomness: an n-sided dice |
| `items.py` | `Weapon` | Items a character can hold; the damage is decided by a dice |
| `character.py` | `Character`, `Warrior`, `Mage` | The characters, with the specialisations as subclasses |
| `battle.py` | `Battle` | Turn-based combat between two characters |

How the classes relate to each other (an arrow means "is used by" or "is the parent of"):


## Setup

All random behaviour goes through Python's `random` module, so we seed it to make the results reproducible. The helper `show_class()` prints an overview of a class (constructor, attributes, properties and methods) and marks what is inherited; we use it to present each class below.

In [1]:
import inspect
import random
from collections import Counter

import matplotlib.pyplot as plt
from pathlib import Path
import sys

project_dir = Path.cwd()
if not (project_dir / "Scripts").is_dir():
    project_dir = project_dir.parent

sys.path.insert(0, str(project_dir / "Scripts"))

In [2]:
from Task_5_Battles import Battle
from Task_1_4_Characters import Character, Mage, Warrior
from Task_2_Dice import Dice
from Task_3_items import Weapon

random.seed(42)  # reproducible results

In [ ]:
def show_class(cls, instance=None) -> None:
    """Print an overview of a class: constructor, attributes, properties and methods."""
    # Collect everything defined in the class or in its parents (not `object`).
    # Parents first, so that a subclass overrides what it redefines.
    owners = {}
    for klass in reversed(cls.__mro__[:-1]):
        for name, member in vars(klass).items():
            owners[name] = (klass, member)

    def label(name: str) -> str:
        klass = owners[name][0]
        return name if klass is cls else f"{name}  (inherited from {klass.__name__})"

    def section(title: str, names: list) -> None:
        if names:
            print(f"  {title}:")
            for name in names:
                print(f"    - {label(name)}")

    bases = [base.__name__ for base in cls.__bases__ if base is not object]
    print(f"class {cls.__name__}" + (f"({', '.join(bases)})" if bases else ""))
    try:
        signature = inspect.signature(cls, eval_str=True)
    except Exception:  # fall back if a type hint cannot be evaluated
        signature = inspect.signature(cls)
    print(f"  constructor: {cls.__name__}{signature}")

    if instance is not None:
        print("  instance attributes (a leading _ means protected):")
        print("    " + ", ".join(vars(instance)))

    constants = [n for n in owners if n.isupper()]
    if constants:
        print("  class constants:")
        for name in constants:
            print(f"    - {name} = {owners[name][1]}")

    section("properties (read-only)", [n for n, (_, m) in owners.items() if isinstance(m, property)])
    section("methods", [n for n, (_, m) in owners.items() if inspect.isfunction(m) and not n.startswith("_")])
    section(
        "special methods",
        [n for n, (_, m) in owners.items() if inspect.isfunction(m) and n.startswith("__") and n != "__init__"],
    )

In [ ]:
# Chart style: light surface, recessive hairline grid, text in ink colours.
# The two series colours (blue, orange) are the first two slots of a colour-blind-safe categorical palette.
SURFACE, INK, INK_2 = "#fcfcfb", "#0b0b0b", "#52514e"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"
BLUE, ORANGE = "#2a78d6", "#eb6834"


def style_axes(ax, grid_axis: str = "y") -> None:
    """Apply a clean, quiet chart style to a matplotlib axes."""
    ax.figure.set_facecolor(SURFACE)
    ax.set_facecolor(SURFACE)
    for side in ("top", "right", "left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.grid(axis=grid_axis, color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    ax.tick_params(colors=INK_2, length=0)
    ax.xaxis.label.set_color(INK_2)
    ax.yaxis.label.set_color(INK_2)

## `Dice` (task 2)

`Dice` models an n-sided dice and is what introduces randomness into the game.

- The number of sides must be at least 2, otherwise a `ValueError` is raised.
- `sides` is stored in the protected attribute `_sides` and exposed through a read-only property, so the validation cannot be bypassed after the dice has been created.
- `roll()` returns `random.randint(1, sides)`. `__call__` lets us write `d20()` instead of `d20.roll()`, and `__repr__` gives strings like `Dice(sides=6)`.